# Academic Early-Intervention: resumable Colab run

Upload two archives to `MyDrive/AcademicEarlyIntervention/`: `Academic-Early-Intervention-Agent.zip` (with a top-level project folder, the existing `results/` checkpoints, source, and requirements) and `Student Performance and Behaviour.zip` (containing the CSV). This notebook expands both under Drive and runs there so checkpoints survive runtime disconnects.

Run the setup cells once, then re-run the training cell after each time-budget stop or reconnect. The project code currently uses CUDA for XGBoost, CatBoost, MLP, and FT-Transformer; LightGBM remains on CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil
import zipfile

DRIVE_FOLDER = Path('/content/drive/MyDrive/AcademicEarlyIntervention')
PROJECT_ARCHIVE = DRIVE_FOLDER / 'Academic-Early-Intervention-Agent.zip'
DATASET_ARCHIVE = DRIVE_FOLDER / 'Student Performance and Behaviour.zip'
WORKSPACE = DRIVE_FOLDER / 'workspace'
PROJECT_DIR = WORKSPACE / 'Academic-Early-Intervention-Agent'
CSV_PATH = PROJECT_DIR / 'data' / 'raw' / 'Student Performance and Behaviour.csv'
DATASET_EXTRACT = WORKSPACE / 'dataset_unpacked'


def safe_extract(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    with zipfile.ZipFile(archive) as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(base):
                raise ValueError(f'Unsafe archive path: {member.filename}')
        zf.extractall(destination)


WORKSPACE.mkdir(parents=True, exist_ok=True)
if not PROJECT_DIR.exists():
    if not PROJECT_ARCHIVE.exists():
        raise FileNotFoundError(f'Upload the project archive to {PROJECT_ARCHIVE}')
    safe_extract(PROJECT_ARCHIVE, WORKSPACE)
PIPELINE_DRIVER = PROJECT_DIR / 'src' / 'pipeline' / 'run_pipeline.py'
if not PIPELINE_DRIVER.exists():
    raise FileNotFoundError(f'Project archive must contain {PROJECT_DIR.name}/src/pipeline/run_pipeline.py')

if not CSV_PATH.exists():
    if not DATASET_ARCHIVE.exists():
        raise FileNotFoundError(f'Upload the dataset archive to {DATASET_ARCHIVE}')
    safe_extract(DATASET_ARCHIVE, DATASET_EXTRACT)
    candidates = list(DATASET_EXTRACT.rglob('Student Performance and Behaviour.csv'))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Expected one CSV in the dataset archive; found {len(candidates)}')
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(candidates[0], CSV_PATH)

print('Project:', PROJECT_DIR)
print('Dataset:', CSV_PATH, f'({CSV_PATH.stat().st_size:,} bytes)')

In [ ]:
# Install the core app/pipeline packages and optional model dependencies.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       '-r', str(PROJECT_DIR / 'requirements.txt'),
                       '-r', str(PROJECT_DIR / 'requirements-extras.txt')])
# Match the recorded metric/data stack where these versions are available.
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'scikit-learn==1.7.1', 'numpy==2.3.1', 'pandas==2.3.1'])
print('Python:', sys.version)

In [ ]:
# Confirm that this runtime actually has a CUDA GPU before requesting --device cuda.
import subprocess, torch
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
import importlib.metadata as metadata
for package in ('xgboost', 'lightgbm', 'catboost', 'torch', 'scikit-learn', 'numpy', 'pandas'):
    print(f'{package}:', metadata.version(package))
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    subprocess.run(['nvidia-smi'], check=False)
    raise RuntimeError('Enable a GPU runtime in Colab, reconnect, and rerun setup before training.')

## Run or resume

This cell runs one time-budget chunk of the full default pipeline. Re-run it to continue. Exit code 3 means the time budget ended cleanly and checkpoints are available; exit code 0 means the full run completed. Keep using the same Drive project and `--device cuda` for every continuation.

In [ ]:
import subprocess, sys
TIME_BUDGET_MINUTES = 45
RUN_LOG = DRIVE_FOLDER / 'colab_pipeline.log'
command = [sys.executable, 'src/pipeline/run_pipeline.py', '--device', 'cuda',
           '--time-budget', str(TIME_BUDGET_MINUTES)]
print('Running:', ' '.join(command), '\nWorking directory:', PROJECT_DIR)
with RUN_LOG.open('a', encoding='utf-8') as log_file:
    completed = subprocess.run(command, cwd=PROJECT_DIR, stdout=log_file,
                               stderr=subprocess.STDOUT, text=True)
tail = RUN_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-50:]
print('\n'.join(tail))
if completed.returncode == 3:
    print('Chunk stopped at the time budget. Re-run this cell to resume.')
elif completed.returncode == 0:
    print('Pipeline completed. Run the export cell to package results and figures.')
else:
    raise RuntimeError(f'Pipeline failed with exit code {completed.returncode}; inspect {RUN_LOG}')

In [ ]:
# Snapshot the current results/ and figures/ under Drive, then offer the zip for download.
from datetime import datetime
from google.colab import files

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPORT_DIR = DRIVE_FOLDER / 'colab_exports' / f'run_{stamp}'
EXPORT_DIR.mkdir(parents=True, exist_ok=False)
for folder_name in ('results', 'figures'):
    source = PROJECT_DIR / folder_name
    if not source.exists():
        raise FileNotFoundError(source)
    shutil.copytree(source, EXPORT_DIR / folder_name)

# Include first-seed fitted artifacts for the five added models when present.
new_model_artifacts = EXPORT_DIR / 'models' / 'pipeline'
new_model_artifacts.mkdir(parents=True, exist_ok=True)
for model in ('xgb', 'lgbm', 'catboost', 'mlp', 'ft_transformer'):
    source = PROJECT_DIR / 'models' / 'pipeline' / f'{model}.joblib'
    if source.exists():
        shutil.copy2(source, new_model_artifacts / source.name)
if not any(new_model_artifacts.iterdir()):
    shutil.rmtree(new_model_artifacts.parent)

missing = []
for model in ('majority', 'midterm_lr', 'logreg', 'logreg_balanced', 'rf_capped',
              'rf_unconstrained', 'hgb', 'xgb', 'lgbm', 'catboost', 'mlp', 'ft_transformer'):
    for seed in range(5):
        stem = f'{model}_seed{seed}'
        for filename in (f'{stem}.json', f'{stem}_val_predictions.npz', f'{stem}_test_predictions.npz'):
            if not (PROJECT_DIR / 'results' / 'runs' / filename).exists():
                missing.append(filename)
if missing:
    print(f'Checkpoint export (not complete): {len(missing)} model-run files are still missing.')
else:
    print('All 12 models have five saved seed runs and prediction files.')

metadata_text = (f'Python: {sys.version}\n'
                 f'PyTorch: {torch.__version__}\n'
                 f'CUDA build: {torch.version.cuda}\n'
                 f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "unavailable"}\n'
                 'Pipeline command: --device cuda\n')
(EXPORT_DIR / 'run_metadata.txt').write_text(metadata_text, encoding='utf-8')
BUNDLE = DRIVE_FOLDER / f'colab_results_figures_{stamp}.zip'
with zipfile.ZipFile(BUNDLE, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder_name in ('results', 'figures', 'models'):
        for path in (EXPORT_DIR / folder_name).rglob('*'):
            if path.is_file():
                zf.write(path, path.relative_to(EXPORT_DIR))
    zf.write(EXPORT_DIR / 'run_metadata.txt', 'run_metadata.txt')
print('Drive export:', EXPORT_DIR)
print('Download bundle:', BUNDLE)
files.download(str(BUNDLE))